# Project (30 pts): Contract on the maximum of two stocks

Carlos Chipantiza 

## Statement

Suppose $S_1(t)$ and $S_2(t)$ are independent stock processes with equal volatility, $S_1(0)=S_2(0)=100$. The contract at maturity pays the value of the most expensive stock: $V(T)=\max\{S_1(T),S_2(T)\}$.

1. What is the formula for the fair value of the contract, $V(0)$?
2. Derive the partial differential equation for this contract.
3. Solve the resultant PDE using the Finite Difference Method.
4. Compare the results obtained through solving the PDE with the Monte Carlo Method.

In [8]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Parameters (not given in the statement)
S1_0 = 100.0    # initial price of stock 1
S2_0 = 100.0    # initial price of stock 2
sigma = 0.3     # volatility of both stocks
r = 0.05        # risk-free rate
T = 1.0         # maturity in years

## 1. What is the formula for the fair value of the contract, $V(0)$?

### Remark: Previous Results

The numbers in brackets refer to my lecture notes; the cells refer to the lecture notebooks.

**(R1) Brownian motion** (Lecture 1). For every $t>0$, $W_t\sim N(0,t)$. Hence $W_T=\sqrt{T}\,Z$ with $Z\sim N(0,1)$.

**(R2) Independent Gaussian variables** (Lecture 1). If $X$ and $Y$ are independent Gaussian random variables, then $X-Y$ is Gaussian and

$$
\operatorname{Var}(X-Y)=\operatorname{Var}(X)+\operatorname{Var}(Y).
$$

**(R3) Solution of the GBM equation** (Lecture 1). The solution of $dS_t=\mu S_t\,dt+\sigma S_t\,dW_t$ is

$$
S_T=S_0\exp\Big(\big(\mu-\tfrac12\sigma^2\big)T+\sigma W_T\Big).
$$

**(R4) Risk-neutral dynamics** (Lecture 3). Under the risk-neutral measure $Q$ the drift $\mu$ is replaced by the risk-free rate $r$:

$$
dS_t=rS_t\,dt+\sigma S_t\,dW_t^Q .
$$

**(R5) Risk-neutral valuation** (Lecture 3). The fair value of a derivative with payoff $h$ at time $T$ is the discounted expected payoff under $Q$:

$$
V(0)=e^{-rT}\,\mathbb{E}^Q[h].
$$

We apply it to the payoff $h=\max\{S_1(T),S_2(T)\}$.

**(R6) Standard normal distribution** (Lecture 3). The density and the distribution function of $Z\sim N(0,1)$ are

$$
\varphi(z)=\frac{1}{\sqrt{2\pi}}\,e^{-z^2/2},
\qquad
N(x)=P(Z\le x),
$$

and $N(-x)=1-N(x)$ for every $x\in\mathbb{R}$.

**(R7) Completing the square** (Lecture 3, proof of the Black–Scholes formula of cell 37; Lemma 6.1, Step 4). For all $a,z\in\mathbb{R}$,

$$
e^{az-\frac12a^2}\,\varphi(z)=\varphi(z-a).
$$

In the lecture it is used with $a=\sigma\sqrt{T}$ to obtain $\mathbb{E}\big[S_T\,\mathbf{1}_{\{S_T>K\}}\big]=S_0e^{rT}N(d_1)$.

**(R8) Independence** (basic probability). If $Z_1$ and $Z_2$ are independent, an expectation $\mathbb{E}[g(Z_1,Z_2)]$ can be computed by first fixing $Z_2=z_2$, computing the expectation in $Z_1$, and then taking the expectation in $Z_2$.

## Solution:

By the risk-neutral valuation formula (R5), the fair value of the contract is

$$
V(0)=e^{-rT}\mathbb{E}^Q
\left[\max\{S_1(T),S_2(T)\}\right].
$$

Under the risk-neutral measure $Q$, both stocks follow geometric Brownian motions with drift $r$ (R4). Using the GBM representation (R3) and the distribution of Brownian motion (R1), we obtain

$$
S_i(T)=100e^{(r-\frac12\sigma^2)T+\sigma\sqrt{T}Z_i},
\qquad i=1,2,
$$

where $Z_1,Z_2\sim N(0,1)$ are independent standard normal random variables, since the two stock processes are independent.

By the independence of $Z_1$ and $Z_2$, their difference satisfies (R2)

$$
Z_1-Z_2\sim N(0,2),
$$

and consequently,

$$
\frac{Z_1-Z_2}{\sqrt{2}}\sim N(0,1).
$$

Since both stocks have the same initial price and volatility, we have

$$
\begin{aligned}
S_1(T)>S_2(T)
&\iff
100e^{(r-\frac12\sigma^2)T+\sigma\sqrt{T}Z_1}
>
100e^{(r-\frac12\sigma^2)T+\sigma\sqrt{T}Z_2}\\
&\iff e^{\sigma\sqrt{T}Z_1}
>
e^{\sigma\sqrt{T}Z_2}\\
&\iff \sigma\sqrt{T}Z_1
>
\sigma\sqrt{T}Z_2\\
&\iff Z_1>Z_2,
\end{aligned}
$$

assuming $\sigma>0$ and $T>0$. In the second step we divide by $100e^{(r-\frac12\sigma^2)T}>0$, in the third we take $\ln$, which is increasing, and in the last we divide by $\sigma\sqrt{T}>0$.

Therefore, splitting according to which stock has the higher price at maturity, we can write

$$
\begin{aligned}
\max\{S_1(T),S_2(T)\}
={}&S_1(T)\mathbf{1}_{\{Z_1>Z_2\}}\\
&+S_2(T)\mathbf{1}_{\{Z_2>Z_1\}},
\end{aligned}
$$

except on the event $\{Z_1=Z_2\}$, which has probability zero.

Since $(Z_1,Z_2)$ and $(Z_2,Z_1)$ have the same joint distribution, and $S_1(T)$ and $S_2(T)$ are the same function of $Z_1$ and $Z_2$ (same initial price and volatility), the two terms have the same expectation. Hence,

$$
\mathbb{E}^Q[\max\{S_1(T),S_2(T)\}]
=
2\mathbb{E}^Q[
S_1(T)\mathbf{1}_{\{Z_1>Z_2\}}
].
$$

Let $a=\sigma\sqrt{T}$. We can rewrite $S_1(T)$ as

$$
S_1(T)=100e^{rT}e^{aZ_1-\frac12a^2}.
$$

Using independence and iterated expectation (R8), we first condition on $Z_2$. For a fixed value $Z_2=z_2$, we have

$$
\begin{aligned}
&\mathbb{E}\left[
e^{aZ_1-\frac12a^2}
\mathbf{1}_{\{Z_1>z_2\}}
\right]\\
&=\int_{z_2}^{\infty}
e^{az-\frac12a^2}\varphi(z)\,dz
\qquad \text{(R6)}\\
&=\int_{z_2}^{\infty}\varphi(z-a)\,dz
\qquad \text{(R7)}\\
&=\int_{z_2-a}^{\infty}\varphi(u)\,du
\qquad (u=z-a)\\
&=P(Z_1+a>z_2).
\end{aligned}
$$

Taking the expectation over $Z_2$, we obtain

$$
\begin{aligned}
&\mathbb{E}\left[
e^{aZ_1-\frac12a^2}
\mathbf{1}_{\{Z_1>Z_2\}}
\right]\\
&=P(Z_1+a>Z_2).
\end{aligned}
$$

Now, using the distribution of the difference of independent standard normal variables (R2),

$$
\begin{aligned}
P(Z_1+a>Z_2)
&=P(Z_1-Z_2>-a)\\
&=P\left(
\frac{Z_1-Z_2}{\sqrt{2}}
>
-\frac{a}{\sqrt{2}}
\right)\\
&=1-N\left(-\frac{a}{\sqrt{2}}\right)\\
&=N\left(\frac{a}{\sqrt{2}}\right),
\end{aligned}
$$

where we have used the symmetry property of the standard normal distribution (R6).

Since $S_1(T)=100e^{rT}e^{aZ_1-\frac12a^2}$, this gives

$$
\mathbb{E}^Q\left[S_1(T)\mathbf{1}_{\{Z_1>Z_2\}}\right]=100e^{rT}N\left(\frac{a}{\sqrt{2}}\right).
$$

Finally, substituting the previous results into the risk-neutral valuation formula, we obtain

$$
\begin{aligned}
V(0)
&=e^{-rT}\cdot
2\mathbb{E}^Q[
S_1(T)\mathbf{1}_{\{Z_1>Z_2\}}
]\\
&=e^{-rT}\cdot
2\cdot100e^{rT}
N\left(\frac{a}{\sqrt{2}}\right)\\
&=200N\left(\frac{\sigma\sqrt{T}}{\sqrt{2}}\right).
\end{aligned}
$$

Therefore, the fair value of the contract is

$$
V(0)=200N\left(\sigma\sqrt{\frac{T}{2}}\right)
$$

where $N(x)$ denotes the cumulative distribution function of a standard normal random variable.

This formula uses that both stocks start at the same price, $S_1(0)=S_2(0)=100$, and have the same volatility $\sigma$.

Notice that the fair value does not depend on the risk-free interest rate $r$, since the discount factor $e^{-rT}$ cancels the growth factor $e^{rT}$ of the stock under the risk-neutral measure.

Now, let's estimate one result with the goal of test our deduction is correct. Note that, if $\sigma=0.3$ and $T=1$, then

$$
V(0)=200N\left(\frac{0.3}{\sqrt{2}}\right)
\approx 116.80.
$$



Now, we evaluate the formula with the parameters given in the example

In [9]:
# V(0) = 2 * S0 * N(sigma * sqrt(T / 2))

def price_formula(S0, sigma, T):
    return 2 * S0 * norm.cdf(sigma * np.sqrt(T / 2))


V_formula = price_formula(S1_0, sigma, T)

print(f"V(0) = {V_formula:.6f}")

V(0) = 116.799597


## 2. Derive the partial differential equation for this contract.

## Previous Results

**(R9) Itô multiplication rules** (Lecture 1; Theorem 7.1).

$$
(dt)^2=0,\qquad dt\,dW_t=0,\qquad (dW_t)^2=dt .
$$

**(R10) Two independent Brownian motions** (Lecture 1; Example 4.16). If $W_1$ and $W_2$ are independent Brownian motions, then

$$
dW_1\,dW_2=0 .
$$

**(R11) Itô's lemma** (Lecture 1; Theorem 7.2). $df$ is obtained from the Taylor expansion of $f$ up to second order, using the multiplication rules (R9). For a function of two stocks we expand in the same way and also use (R10).

**(R12) Riskless portfolio** (Lecture 3, cells 34–36; Proposition 5.2 and Theorem 5.1). The delta-hedged portfolio, with $\Delta=\partial f/\partial S$ held fixed over $[t,t+dt]$, has no $dW$ term. By no-arbitrage, a riskless portfolio earns the risk-free rate: $d\Pi=r\Pi\,dt$.

**(R13) Terminal and boundary conditions** (Lecture 3, cell 37; Remark 5.6). The payoff is the terminal condition $f(T,S)=h(S)$, and a stock whose price is $0$ stays at $0$.

## Solution: 


We derive the pricing PDE by extending the delta-hedging argument from Lecture 3 to a contract written on two underlying stocks.

We assume that the two stocks follow independent geometric Brownian motions with constant volatilities.

**Dynamics of the two stocks**

Let $S_1$ and $S_2$ denote the prices of the two underlying stocks. Their dynamics are given by

$$
dS_i=\mu_iS_i\,dt+\sigma S_i\,dW_i,
\qquad i=1,2,
$$

where $W_1$ and $W_2$ are independent Brownian motions. We assume that both stocks have the same constant volatility $\sigma$, although their expected returns $\mu_1$ and $\mu_2$ may differ.

Let $V=V(t,S_1,S_2)$ denote the value of the contract.

**Products of the stock increments**

Using the Itô multiplication rules (R9), we first calculate the square of each stock increment:

$$
\begin{aligned}
(dS_i)^2
&=\left(\mu_iS_i\,dt+\sigma S_i\,dW_i\right)^2\\
&=\mu_i^2S_i^2(dt)^2
+2\mu_i\sigma S_i^2\,dt\,dW_i
+\sigma^2S_i^2(dW_i)^2\\
&=\sigma^2S_i^2\,dt.
\end{aligned}
$$

Next, since the Brownian motions $W_1$ and $W_2$ are independent, their quadratic covariation is zero (R10). Therefore, since the terms with $dt$ vanish by (R9),

$$
\begin{aligned}
dS_1\,dS_2
&=\left(\mu_1S_1\,dt+\sigma S_1\,dW_1\right)
\left(\mu_2S_2\,dt+\sigma S_2\,dW_2\right)\\
&=\sigma^2S_1S_2\,dW_1\,dW_2\\
&=0.
\end{aligned}
$$

Thus, the quadratic variation terms are

$$
(dS_1)^2=\sigma^2S_1^2\,dt,
\qquad
(dS_2)^2=\sigma^2S_2^2\,dt,
\qquad
dS_1\,dS_2=0.
$$

**Apply Itô's lemma**

Applying the two-variable version of Itô's lemma (R11) to $V(t,S_1,S_2)$, we obtain

$$
\begin{aligned}
dV={}&
\frac{\partial V}{\partial t}\,dt
+\frac{\partial V}{\partial S_1}\,dS_1
+\frac{\partial V}{\partial S_2}\,dS_2 +\frac12\frac{\partial^2V}{\partial S_1^2}(dS_1)^2
+\frac12\frac{\partial^2V}{\partial S_2^2}(dS_2)^2 +\frac{\partial^2V}{\partial S_1\partial S_2}\,dS_1\,dS_2.
\end{aligned}
$$

Substituting the quadratic variation terms obtained in Step 2 gives

$$
\begin{aligned}
dV={}&
\left(
\frac{\partial V}{\partial t}
+\frac12\sigma^2S_1^2\frac{\partial^2V}{\partial S_1^2}
+\frac12\sigma^2S_2^2\frac{\partial^2V}{\partial S_2^2}
\right)dt +\frac{\partial V}{\partial S_1}\,dS_1
+\frac{\partial V}{\partial S_2}\,dS_2.
\end{aligned}
$$

Notice that the mixed second-order derivative disappears because $dS_1\,dS_2=0$.

**Construct a riskless portfolio**

Following the delta-hedging argument from Lecture 3 (R12), we construct a portfolio consisting of:

- a short position in one unit of the derivative;
- a long position of $\Delta_1$ shares of stock 1;
- a long position of $\Delta_2$ shares of stock 2.

The portfolio value is

$$
\Pi=-V+\Delta_1S_1+\Delta_2S_2,
$$

where the hedge ratios are chosen as

$$
\Delta_1=\frac{\partial V}{\partial S_1},
\qquad
\Delta_2=\frac{\partial V}{\partial S_2}.
$$

Over the infinitesimal interval $[t,t+dt]$, the hedge ratios are held fixed. Consequently,

$$
d\Pi=-dV+\Delta_1\,dS_1+\Delta_2\,dS_2.
$$

Substituting the expression for $dV$ from Step 3, the stochastic terms cancel:

$$
\begin{aligned}
d\Pi
={}&-
\left(
\frac{\partial V}{\partial t}
+\frac12\sigma^2S_1^2\frac{\partial^2V}{\partial S_1^2}
+\frac12\sigma^2S_2^2\frac{\partial^2V}{\partial S_2^2}
\right)dt.
\end{aligned}
$$

Therefore, the portfolio is locally riskless because it contains no Brownian increments.

**Apply the no-arbitrage condition**

By the no-arbitrage principle (R12), a locally riskless portfolio must earn the risk-free rate $r$. Hence,

$$
d\Pi=r\Pi\,dt.
$$

Using the expression for the portfolio value, we obtain

$$
d\Pi
=r\left(
-V+S_1\frac{\partial V}{\partial S_1}
+S_2\frac{\partial V}{\partial S_2}
\right)dt.
$$

Equating this expression with the result obtained in Step 4 gives

$$
\begin{aligned}
-&\left(
\frac{\partial V}{\partial t}
+\frac12\sigma^2S_1^2\frac{\partial^2V}{\partial S_1^2}
+\frac12\sigma^2S_2^2\frac{\partial^2V}{\partial S_2^2}
\right) =r\left(
-V+S_1\frac{\partial V}{\partial S_1}
+S_2\frac{\partial V}{\partial S_2}
\right).
\end{aligned}
$$

Rearranging the terms yields the required pricing equation.

**The pricing PDE**

The resulting partial differential equation is

$$
\begin{aligned}
\frac{\partial V}{\partial t}
&+\frac12\sigma^2S_1^2
\frac{\partial^2V}{\partial S_1^2}
+\frac12\sigma^2S_2^2
\frac{\partial^2V}{\partial S_2^2} +rS_1\frac{\partial V}{\partial S_1}
+rS_2\frac{\partial V}{\partial S_2}
-rV=0.
\end{aligned}
$$

This equation holds for

$$
0\leq t<T,\qquad S_1>0,\quad S_2>0.
$$

An important observation is that the expected returns $\mu_1$ and $\mu_2$ do not appear in the final PDE. This is a consequence of the delta-hedging argument: the stochastic exposure is eliminated, and the resulting riskless portfolio must earn the risk-free rate.

**Terminal and boundary conditions**

The contract pays the maximum of the two stock prices at maturity. Therefore, its terminal condition is

$$
V(T,S_1,S_2)=\max\{S_1,S_2\}.
$$

Furthermore, a stock whose price reaches zero remains at zero (R13).

If $S_1=0$, the payoff at maturity becomes $S_2(T)$. Thus, the contract is equivalent to holding one unit of stock 2:

$$
V(t,0,S_2)=S_2.
$$

Similarly, if $S_2=0$, the contract is equivalent to holding one unit of stock 1:

$$
V(t,S_1,0)=S_1.
$$

These are the terminal and boundary conditions required to complete the pricing problem.

**Observation**

The mixed derivative disappears because the two Brownian motions are independent. If they were correlated with correlation coefficient $\rho$, the PDE would contain the additional term

$$
\rho\sigma^2S_1S_2
\frac{\partial^2V}{\partial S_1\partial S_2}.
$$




## 3. Solve the resultant PDE using the Finite Difference Method.

## Previous Results

**(R14) Finite differences** (Lecture 7). By Taylor's theorem, the forward and backward differences have error $O(h)$ and the central difference has error $O(h^2)$:

$$
f'(x)\approx\frac{f(x+h)-f(x-h)}{2h},
\qquad
f''(x)\approx\frac{f(x+h)-2f(x)+f(x-h)}{h^2}.
$$

**(R15) The grid** (Lecture 7). The values of $V$ are approximated on the nodes $S_i=S_{\min}+i\,\Delta S$, $t_j=j\,\Delta t$, with $w_{i,j}\approx V(S_i,t_j)$. The terminal condition gives the values at $t=T$, and the scheme moves backward in time until $t=0$.

**(R16) Explicit scheme for the Black–Scholes PDE in $S$** (Lecture 7). With a backward difference in time and central differences in $S$,

$$
w_{i,j-1}=a_i\,w_{i-1,j}+b_i\,w_{i,j}+c_i\,w_{i+1,j},
$$

$$
a_i=\tfrac12\Delta t\Big(\sigma^2\big(\tfrac{S_i}{\Delta S}\big)^2-r\,\tfrac{S_i}{\Delta S}\Big),
\qquad
b_i=1-\Delta t\Big(\sigma^2\big(\tfrac{S_i}{\Delta S}\big)^2+r\Big),
\qquad
c_i=\tfrac12\Delta t\Big(\sigma^2\big(\tfrac{S_i}{\Delta S}\big)^2+r\,\tfrac{S_i}{\Delta S}\Big).
$$

**(R17) Stability of the explicit scheme** (Lecture 7). The weights add up to $a_i+b_i+c_i=1-r\Delta t$. If all of them are nonnegative, each new value is an average of old values and the scheme cannot blow up. The binding condition is $b_i\ge 0$, which fails first at the largest $S_i$; if it fails, the errors grow at every step.

**(R18) Boundary conditions on a truncated grid** (Lecture 7). The grid stops at $S_{\max}$, where a boundary value is needed. For the call, $V(S_{\max},t)=S_{\max}-Ke^{-r(T-t)}$, which is linear in $S$; hence $\partial^2V/\partial S^2=0$ there.

## Solution:



We now solve the pricing PDE derived in Section 2 using the explicit finite difference method. Following Lecture 7, we discretize the time and stock-price dimensions and approximate the derivatives using finite differences. Starting from the terminal payoff, the numerical solution is then computed backward in time until the initial contract value is obtained.

**Construction of the grid (R15)**

We consider a two-dimensional spatial grid because the contract depends on the prices of both underlying stocks. Let $S_{\max}$ denote the upper limit of the computational domain, and divide each stock-price interval into $N$ equally spaced subdivisions. Thus,

$$
S_{1,i}=i\Delta S,\qquad
S_{2,k}=k\Delta S,
\qquad i,k=0,\ldots,N,
$$

where

$$
\Delta S=\frac{S_{\max}}{N}.
$$

Similarly, we divide the time interval $[0,T]$ into $M$ equally spaced time steps:

$$
t_j=j\Delta t,\qquad j=0,\ldots,M,
\qquad
\Delta t=\frac{T}{M}.
$$

We denote the numerical approximation of the contract value at each grid point by

$$
w^j_{i,k}\approx V(t_j,S_{1,i},S_{2,k}).
$$

The objective is to determine the value at $t=0$. To do so, we begin with the known terminal payoff at $t=T$ and apply the numerical scheme recursively, moving backward through the time grid.

**Terminal condition**

At maturity, the contract pays the maximum of the two stock prices. Therefore, the terminal condition

$$
V(T,S_1,S_2)=\max\{S_1,S_2\}
$$

provides the values at the final time level:

$$
w^M_{i,k}=\max\{S_{1,i},S_{2,k}\}.
$$

These values serve as the starting point for the backward numerical procedure.

**Derivation of the explicit finite difference scheme (R14, R16)**

Recall that the pricing PDE obtained in Section 2 is

$$
\begin{aligned}
\frac{\partial V}{\partial t}
&+\frac12\sigma^2S_1^2
\frac{\partial^2V}{\partial S_1^2}
+\frac12\sigma^2S_2^2
\frac{\partial^2V}{\partial S_2^2} +rS_1\frac{\partial V}{\partial S_1}
+rS_2\frac{\partial V}{\partial S_2}
-rV=0.
\end{aligned}
$$

To approximate its solution numerically, we use a backward difference for the time derivative and central differences for the spatial derivatives, following the finite difference results from Lecture 7 (R14).

At a grid point $(t_j,S_{1,i},S_{2,k})$, the time derivative is approximated by

$$
\frac{\partial V}{\partial t}
\approx
\frac{w^j_{i,k}-w^{j-1}_{i,k}}{\Delta t}.
$$

For the derivatives with respect to the first stock price, we use

$$
\frac{\partial V}{\partial S_1}
\approx
\frac{w^j_{i+1,k}-w^j_{i-1,k}}{2\Delta S},
$$

and

$$
\frac{\partial^2V}{\partial S_1^2}
\approx
\frac{w^j_{i+1,k}-2w^j_{i,k}+w^j_{i-1,k}}
{(\Delta S)^2}.
$$

The corresponding approximations for the second stock price are

$$
\frac{\partial V}{\partial S_2}
\approx
\frac{w^j_{i,k+1}-w^j_{i,k-1}}{2\Delta S},
$$

and

$$
\frac{\partial^2V}{\partial S_2^2}
\approx
\frac{w^j_{i,k+1}-2w^j_{i,k}+w^j_{i,k-1}}
{(\Delta S)^2}.
$$

Substituting these expressions into the PDE and rearranging to isolate the value at the previous time level, $w^{j-1}_{i,k}$, yields

$$
\begin{aligned}
w^{j-1}_{i,k}
={}&a_iw^j_{i-1,k}
+c_iw^j_{i+1,k} +a_kw^j_{i,k-1}
+c_kw^j_{i,k+1}
+b_{i,k}w^j_{i,k}.
\end{aligned}
$$

Since $S_{1,i}/\Delta S=i$ and $S_{2,k}/\Delta S=k$, the coefficients are given by

$$
a_i=\frac12\Delta t
\left(\sigma^2i^2-ri\right),
\qquad
c_i=\frac12\Delta t
\left(\sigma^2i^2+ri\right),
$$

$$
a_k=\frac12\Delta t
\left(\sigma^2k^2-rk\right),
\qquad
c_k=\frac12\Delta t
\left(\sigma^2k^2+rk\right),
$$

and

$$
b_{i,k}
=1-\Delta t
\left(\sigma^2i^2+\sigma^2k^2+r\right).
$$

The resulting scheme is applied at the interior grid points, where $i,k=1,\ldots,N-1$, and for time levels $j=M,\ldots,1$.

It is worth noting that the discounting term $-rV$ contributes only once to the central coefficient $b_{i,k}$, since it appears only once in the original PDE.

**Boundary conditions (R13, R18)**

To complete the numerical scheme, we must specify the values along the boundaries of the computational domain. At the lower boundaries, we use the conditions derived in Section 2. If the first stock price is zero, the contract is equivalent to holding one unit of the second stock. Conversely, if the second stock price is zero, its value is equal to the first stock price. Hence,

$$
w^j_{0,k}=S_{2,k},
\qquad
w^j_{i,0}=S_{1,i}.
$$

For the upper boundaries, we use linear extrapolation. The motivation is that, when one stock price becomes sufficiently large relative to the other, the maximum payoff is dominated by the larger stock price. Consequently, the contract value becomes approximately linear in that stock-price dimension, and its corresponding second derivative is approximately zero.

Using a central second-difference approximation at the upper boundary, this condition leads to

$$
w^j_{N,k}=2w^j_{N-1,k}-w^j_{N-2,k},
$$

and, analogously,

$$
w^j_{i,N}=2w^j_{i,N-1}-w^j_{i,N-2}.
$$

These conditions provide an approximation to the behaviour of the contract outside the truncated computational domain.

**Stability condition (R17)**

Since we are using an explicit finite difference scheme, the choice of the time step must satisfy a stability restriction. Following the criterion discussed in Lecture 7 (R17), we require all five coefficients in the update formula to be nonnegative:

$$
a_i,c_i,a_k,c_k,b_{i,k}\geq0.
$$

For the lower-neighbour coefficients, we have

$$
a_i\geq0
\iff
\sigma^2i^2-ri\geq0
\iff
\sigma^2i\geq r,
$$

and similarly,

$$
a_k\geq0
\iff
\sigma^2k\geq r.
$$

The upper-neighbour coefficients $c_i$ and $c_k$ are nonnegative for positive $r$, $\sigma$, $i$, and $k$.

The remaining restriction comes from the central coefficient. Requiring $b_{i,k}\geq0$ gives

$$
\Delta t
\left(\sigma^2i^2+\sigma^2k^2+r\right)
\leq1.
$$

The coefficient $b_{i,k}$ decreases as $i$ and $k$ grow, and the scheme is applied for $i,k\le N-1$, so the most restrictive point is $i=k=N-1$. Taking $i=k=N$ gives a slightly stronger condition, which is therefore sufficient for all grid points:

$$
\Delta t\left(2\sigma^2N^2+r\right)\leq1.
$$

Using $\Delta t=T/M$, we obtain the following restriction on the number of time steps:

$$
M\geq T\left(2\sigma^2N^2+r\right).
$$

In practice, we can choose

$$
M=
\left\lceil
T\left(2\sigma^2N^2+r\right)
\right\rceil,
$$

or a larger integer to ensure that the stability condition is satisfied.

For the parameters considered in this problem, $\sigma=0.3$ and $r=0.05$, so that

$$
\sigma^2=0.09>r=0.05.
$$

It follows that $a_i$ and $a_k$ are nonnegative for every $i,k\geq1$. Thus, under these parameter values, the sufficient nonnegativity condition is determined by the central coefficient $b_{i,k}$.

Furthermore, the sum of the five coefficients is

$$
\begin{aligned}
a_i+c_i+a_k+c_k+b_{i,k}
&=\Delta t\sigma^2i^2
+\Delta t\sigma^2k^2\\
&\quad+1-\Delta t
\left(\sigma^2i^2+\sigma^2k^2+r\right)\\
&=1-r\Delta t.
\end{aligned}
$$

This identity is consistent with the discounting term in the original pricing PDE.

**Approximation of the initial contract value**

Once the terminal and boundary conditions have been specified, we apply the explicit scheme recursively, starting at $j=M$ and proceeding backward until $j=0$.

The initial contract value is obtained at

$$
t_0=0,\qquad S_1=S_2=100.
$$

Since both spatial grids have the same spacing, the corresponding index is

$$
i^*=\frac{100}{\Delta S}.
$$

Provided that $100$ is exactly a grid point, the numerical approximation to the initial price is therefore

$$
V(0,100,100)\approx w^0_{i^*,i^*}.
$$

We choose $S_{\max}$ and $N$ so that $100$ is a grid point; with $S_{\max}=300$ and $N=60$ we have $\Delta S=5$ and $i^*=20$.

In [10]:

def price_fd(Smax, N):

    # Spatial and time steps
    dS = Smax / N
    M = int(np.ceil(T * (2 * sigma**2 * N**2 + r)))
    dt = T / M

    # Stock price grid
    S1 = np.linspace(0, Smax, N + 1)
    S2 = np.linspace(0, Smax, N + 1)

    # Terminal condition
    w = np.maximum(S1[:, None], S2[None, :])

    # Grid indices
    i = np.arange(1, N)
    k = np.arange(1, N)

    # Coefficients
    a_i = 0.5 * dt * (sigma**2 * i**2 - r * i)
    c_i = 0.5 * dt * (sigma**2 * i**2 + r * i)

    a_k = 0.5 * dt * (sigma**2 * k**2 - r * k)
    c_k = 0.5 * dt * (sigma**2 * k**2 + r * k)

    b = 1 - dt * (
        sigma**2 * i[:, None]**2
        + sigma**2 * k[None, :]**2 + r
    )

    # Backward time iteration
    for j in range(M):

        w_new = np.zeros_like(w)

        # Explicit finite difference scheme
        w_new[1:N, 1:N] = (
            a_i[:, None] * w[0:N-1, 1:N]
            + c_i[:, None] * w[2:N+1, 1:N]
            + a_k[None, :] * w[1:N, 0:N-1]
            + c_k[None, :] * w[1:N, 2:N+1]
            + b * w[1:N, 1:N]
        )

        # Lower boundary conditions
        w_new[0, :] = S2
        w_new[:, 0] = S1

        # Upper boundary conditions
        w_new[N, 1:N] = (
            2 * w_new[N-1, 1:N] - w_new[N-2, 1:N]
        )

        w_new[1:N, N] = (
            2 * w_new[1:N, N-1] - w_new[1:N, N-2]
        )

        # Upper-right corner
        w_new[N, N] = (
            w_new[N, N-1] + w_new[N-1, N]
            - w_new[N-1, N-1]
        )

        w = w_new

    # Initial contract value
    i_star = int(round(S1_0 / dS))
    V0 = w[i_star, i_star]

    return V0, M


# Numerical price and error
V_fd, M = price_fd(300, 60)

print(f"V(0) = {V_fd:.6f} M = {M} error = {V_fd - V_formula:+.6f}")


V(0) = 116.773993 M = 649 error = -0.025604


## 4. Compare the results obtained through solving the PDE with the Monte Carlo Method.

## Previous Results

**(R19) Simulation of a geometric Brownian motion** (Lecture 1). The price at time $T$ can be simulated exactly from one standard normal number $Z$:

$$
S_T=S_0\exp\Big(\big(\mu-\tfrac12\sigma^2\big)T+\sigma\sqrt{T}\,Z\Big),
$$

with $\mu$ replaced by $r$ under $Q$ (R4).

**(R20) Law of large numbers** (basic probability). If $Y^{(1)},Y^{(2)},\dots$ are independent with the same distribution as $Y$, then

$$
\frac1n\sum_{m=1}^{n}Y^{(m)}\longrightarrow\mathbb{E}[Y]\qquad (n\to\infty).
$$

**(R21) Central limit theorem** (Lecture 1). For large $n$, the sample mean is approximately Gaussian with standard deviation $\operatorname{sd}(Y)/\sqrt{n}$; together with $N(1.96)=0.975$ (Lecture 3), this gives the $95\%$ confidence interval

$$
\bar Y_n\pm 1.96\,\frac{s_n}{\sqrt n}.
$$

## Monte Carlo Method

**Simulation of the two stocks at maturity (R19)**

We simulate the two stock prices at maturity using independent standard normal random variables:

$$
S_i^{(m)}(T)=100\exp\left(
\left(r-\frac12\sigma^2\right)T
+\sigma\sqrt{T}Z_i^{(m)}
\right),
$$

where $i=1,2$, $m=1,\ldots,n$, and all the random variables $Z_i^{(m)}$ are independent and follow a standard normal distribution.

**Discounted payoffs**

For each simulation, we calculate the maximum stock price and discount it to the present:

$$
Y^{(m)}=e^{-rT}\max\left\{
S_1^{(m)}(T),S_2^{(m)}(T)
\right\}.
$$

**The Monte Carlo estimator (R5), (R20)**

The Monte Carlo estimator is the average of all discounted payoffs:

$$
\widehat V_n=\frac{1}{n}\sum_{m=1}^{n}Y^{(m)}
$$

By the law of large numbers:

$$
\widehat V_n\xrightarrow[n\to\infty]{a.s.}
e^{-rT}\mathbb{E}^Q\left[
\max\{S_1(T),S_2(T)\}
\right].
$$

**Standard error and confidence interval (R21)**

We calculate the sample variance:

$$
s_n^2=\frac{1}{n-1}\sum_{m=1}^{n}
\left(Y^{(m)}-\widehat V_n\right)^2.
$$

The standard error is:

$$
\mathrm{SE}_n=\frac{s_n}{\sqrt n}.
$$

Using the central limit theorem, an approximate 95% confidence interval is:

$$
\left[
\widehat V_n-1.96\,\mathrm{SE}_n,\;
\widehat V_n+1.96\,\mathrm{SE}_n
\right]
$$

**Rate of convergence**

The standard error decreases proportionally to the inverse square root of the number of simulations:

$$
\mathrm{SE}_n\propto\frac{1}{\sqrt n}
$$

Therefore, increasing the number of simulations improves the accuracy of the Monte Carlo estimator.

In [11]:
# Number of simulations
n = 1000000

# Generate random numbers
np.random.seed(0)
Z1 = np.random.randn(n)
Z2 = np.random.randn(n)

# Simulate stock prices at maturity

S1_T = S1_0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z1)

S2_T = S1_0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z2)

# Discounted payoffs
Y = np.exp(-r * T) * np.maximum(S1_T, S2_T)

# Monte Carlo estimator
V_MC = np.mean(Y)

# Standard error
SE = np.std(Y, ddof=1) / np.sqrt(n)

# 95% confidence interval
lower = V_MC - 1.96 * SE
upper = V_MC + 1.96 * SE

# Results
print("Analytical price:", V_formula)
print("Monte Carlo price:", V_MC)
print("Standard error:", SE)
print("95% confidence interval:", lower, upper)
print("Absolute error:", abs(V_MC - V_formula))
print("Finite difference price:", V_fd)
print("Difference MC - FD:", V_MC - V_fd)
print("Finite difference price inside confidence interval:", lower <= V_fd <= upper)

Analytical price: 116.79959714273636
Monte Carlo price: 116.81595098447603
Standard error: 0.029905189791200038
95% confidence interval: 116.75733681248528 116.87456515646679
Absolute error: 0.016353841739672248
Finite difference price: 116.77399272989229
Difference MC - FD: 0.041958254583747134
Finite difference price inside confidence interval: True


## 5. Comparison and Conclusions

We compare the analytical solution, the Finite Difference Method and the Monte Carlo Method.

The absolute errors are calculated as:

$$
E_{\mathrm{FD}}=|V_{\mathrm{FD}}-V_{\mathrm{formula}}|
$$

$$
E_{\mathrm{MC}}=|V_{\mathrm{MC}}-V_{\mathrm{formula}}|
$$



In [12]:
# Calculate absolute errors
error_FD = abs(V_fd - V_formula)
error_MC = abs(V_MC - V_formula)

# Check the Monte Carlo confidence interval
inside_CI = lower <= V_formula <= upper

# Display comparison table
print(f"{'Method':<22} {'Price':>12} {'Absolute error':>17}")
print("-" * 53)

print(f"{'Analytical':<22} {V_formula:>12.6f} {'-':>17}")
print(f"{'Finite Difference':<22} {V_fd:>12.6f} {error_FD:>17.6f}")
print(f"{'Monte Carlo':<22} {V_MC:>12.6f} {error_MC:>17.6f}")

print("\n Monte Carlo 95% confidence interval:")
print(f"[{lower:.6f}, {upper:.6f}]")

print("\n Analytical price inside confidence interval:", inside_CI)

Method                        Price    Absolute error
-----------------------------------------------------
Analytical               116.799597                 -
Finite Difference        116.773993          0.025604
Monte Carlo              116.815951          0.016354

 Monte Carlo 95% confidence interval:
[116.757337, 116.874565]

 Analytical price inside confidence interval: True


## Conclusions


- The pricing PDE, together with its terminal and boundary conditions, completely specifies the pricing problem for this contract under the stated assumptions.

- The explicit finite difference method converts the two-dimensional pricing PDE into a recursive numerical scheme. By using the terminal payoff as the starting condition, imposing suitable boundary values, and choosing time and space steps that satisfy the stability restriction, we can approximate the contract value at the initial time.

- The finite difference price has a discretization error of about $0.026$ (with $\Delta S=5$), and the Monte Carlo price has a statistical error of about $0.03$ (one standard error with $n=10^6$). The difference between the two numerical prices is of the same order as these errors, so the solution of the PDE and the Monte Carlo estimate agree.


- The three methods give similar results.


- Finite Difference Method: Gives a good approximation. Increasing the number of grid points can improve the result, but requires more calculations. 

- Monte Carlo Method: Also gives a good approximation. Increasing the number of simulations generally makes the result more accurate.

- The Finite Difference Method requires calculations over a grid, while Monte Carlo uses random numbers to estimate the price.

- The analytical price falls within the 95% confidence interval obtained from Monte Carlo, supporting the accuracy of the simulation.

## Use of AI

AI was used for the tasks below. 

- Planning a roadmap of the project steps.
- Summarizing Theory and creating notes of lecture
- Identifying bugs and systanx errors. 